# Clase 5 · Conseguir datos (y poder confiar en ellos) · Clase escrita
### Herramientas computacionales

Esta semana no hay clase por Zoom. Este notebook reemplaza la clase: todo lo que les habría explicado en vivo está escrito en las celdas de texto. Léanlo en orden, ejecuten cada celda y hagan los ejercicios. Dediquenle el mismo tiempo que le hubieran dedicado a la clase para fijar los conocimientos. Cualquier duda quedo al pendiente por Slack.

**Objetivos de hoy**

- Extraer una tabla de una página web eligiendo las filas por su estructura (`tbody tr`) y no por su posición, y explicar por qué lo que muestra el navegador puede no ser lo que se descarga.
- Pedirle datos a una API con `requests` (`params`, `timeout`, `raise_for_status()`), pasar el JSON a un DataFrame y **traer todas las páginas** de una respuesta.
- Extraer una tabla de un PDF real, darse cuenta de cuándo la extracción salió mal, convertir los números (`3.219.016`, `18,4%`, `-`) y **verificar** el resultado contra los totales del propio documento.
- Guardar tablas en SQLite, consultarlas con SQL y encontrar las filas que un `JOIN` pierde sin avisar.
- Reconocer el código de los videos que quedó viejo y saber por qué reemplazarlo.

## 0. Preparación

Esta clase usa cinco librerías que quizás no tengan: `requests`, `beautifulsoup4`, `lxml`, `pypdf` y `pdfplumber`.

**En su computadora**, con el entorno del curso activado (lo vimos en la clase 4):
```
pip install -r Clase5/requirements.txt
```

**En Colab**, en una celda: `%pip install pypdf pdfplumber` (las otras ya vienen instaladas), y suban la carpeta `archivos` con `boletin_ife.pdf` y `provincias_ign.csv`. **En Colab no usen el `requirements.txt`**: intentaría cambiar librerías que Colab trae y necesita, y aparece un aviso que pide reiniciar la sesión.

`sqlite3`, `json` y `re` vienen con Python, es decir que no tienen que instalarlas ni van en el `requirements.txt`.

Para utilizar el paquete beautifulsoup4 se instala `beautifulsoup4` y se importa `bs4`.

In [ ]:
#Si están corriendo este notebook localmente en su computadora:
#pip install -r Clase5/requirements.txt

#Si estan en Colab seguramente tendrán que correr esto:
%pip install pdfplumber
%pip install pypdf

In [ ]:
import importlib
import sys

print("Python", sys.version.split()[0])
for instalar, importar in [("pandas", "pandas"), ("requests", "requests"), ("beautifulsoup4", "bs4"),
                           ("lxml", "lxml"), ("pypdf", "pypdf"), ("pdfplumber", "pdfplumber")]:
    try:
        modulo = importlib.import_module(importar)
        print(f"  {importar:11} {modulo.__version__}")
    except ImportError:
        print(f"  {importar:11} FALTA: pip install {instalar}")

Si `pypdf` aparece con una versión menor a 3, actualícenlo (`pip install -U pypdf`).

## 1. *Web scraping*: lo que se ve y lo que se descarga

**En el video vimos:** qué es un HTML, las etiquetas y sus atributos, `requests` para pedir la página y `BeautifulSoup` para leerla.

Repaso rápido de lo que hay que tener presente:

- **HTML**: texto con **etiquetas** que describe la estructura de una página. Una etiqueta abre y cierra (`<td>…</td>`) y puede tener **atributos** (`href="…"`, `class="…"`, `id="…"`).
- **En una tabla**: `<table>` es la tabla, `<thead>` el encabezado, `<tbody>` el cuerpo, `<tfoot>` el pie; `<tr>` es una fila, `<th>` una celda de encabezado y `<td>` una celda de datos.
- **Selector CSS**: una forma de señalar elementos. `table#example tbody tr` quiere decir «las filas (`tr`) del cuerpo (`tbody`) de la tabla cuyo `id` es `example`».

### 1.1 Lo que se ve no es lo que se descarga

Vuelvan a mirar dos slides del video:

- **Slide 14** (la página en el navegador): la primera fila es **Airi Satou**, y abajo dice *Showing 1 to 10 of 57 entries*.
- **Slide 17** (lo que imprimió `print(info)`): la primera fila es **Tiger Nixon**.

**Predecir.** ¿Por qué el navegador muestra a Airi Satou primero y el HTML trae a Tiger Nixon? ¿Cuántas filas tiene el HTML: 10 o 57?

<details><summary><b>Ver la respuesta</b></summary>

El HTML trae **las 57 filas**, en el orden en que se escribieron (Tiger Nixon primero). En el navegador, un programa de JavaScript **ordena** la tabla por nombre y la **pagina** de a 10. `requests` no ejecuta JavaScript: recibe el HTML tal como lo manda el servidor.

Hay tres casos, y conviene distinguirlos antes de programar:

1. **Los datos están en el HTML** (esta página). `requests` y BeautifulSoup alcanzan.
2. **El navegador los pide después a una API**, con JavaScript. El HTML viene vacío, pero en las herramientas del navegador (**F12 → Red/Network → Fetch/XHR**) se ve la dirección de esa API. Casi siempre conviene usarla directamente (sección 3).
3. **Hace falta hacer clics, completar formularios o iniciar sesión.** Ahí se usan navegadores automatizados (Selenium, Playwright). No lo profundizaremos en el curso, pero si lo quieren usar para su TP Final esta perfecto. Me avisan y los guio para que profundicen en esos paquetes.

Para comparar: en el navegador, hagan clic derecho → seleccionen **Ver código fuente** y les mostrará lo que descarga `requests`; sino, seleccionen **Inspeccionar** y les mostrará el HTML *después* de que corrió el JavaScript.
</details>

### 1.2 Corre, pero falla en silencio: descartar filas por su posición

Esta es una versión chica de la página de la slide 14, con su estructura real: encabezado (`thead`), cuerpo (`tbody`) y pie (`tfoot`, que repite los títulos). Abajo está el código de la slide 18, que descarta la primera y la última fila.

**Predecir.** ¿Cuántas filas tiene el DataFrame? Y si la página se actualiza y **deja de tener pie**, ¿cuántas?

In [ ]:
import pandas as pd
from bs4 import BeautifulSoup

#Creamos dos html para este ejercicio de revisión. Uno llamado html_con_pie y otro html_sin_pie:
encabezado = "<tr><th>Name</th><th>Position</th><th>Office</th><th>Age</th><th>Start date</th><th>Salary</th></tr>"
cuerpo = """
<tr><td>Tiger Nixon</td><td>System Architect</td><td>Edinburgh</td><td>61</td><td>2011/04/25</td><td>$320,800</td></tr>
<tr><td>Garrett Winters</td><td>Accountant</td><td>Tokyo</td><td>63</td><td>2011/07/25</td><td>$170,750</td></tr>
<tr><td>Ashton Cox</td><td>Junior Technical Author</td><td>San Francisco</td><td>66</td><td>2009/01/12</td><td>$86,000</td></tr>
"""
html_con_pie = f'<table id="example"><thead>{encabezado}</thead><tbody>{cuerpo}</tbody><tfoot>{encabezado}</tfoot></table>'
html_sin_pie = f'<table id="example"><thead>{encabezado}</thead><tbody>{cuerpo}</tbody></table>'


def limpia_filas(html):
    """Descarta la primera y la última fila por su posición (El código de la slide 18)"""
    info = BeautifulSoup(html, "html.parser").find_all("tr")
    df = pd.DataFrame(columns=["name", "position", "office", "age", "start", "salary"])
    for i, item in enumerate(info):
        if i != 0 and i != len(info) - 1:
            df.loc[i - 1] = [valor.text for valor in item.find_all("td")]
    return df


print(limpia_filas(html_con_pie).shape)
print(limpia_filas(html_sin_pie).shape)
limpia_filas(html_sin_pie)

Con pie salen las 3 filas. Sin pie, el código **pierde a Ashton Cox sin dar ningún error**: la última fila ya no era el pie, era un dato. Descartar filas por su posición solo funciona cuando conocen muy bien los datos y estan completamente seguros que la tabla siempre tendra esa esctuctura. Sin embargo, como las páginas web cambian, una buena práctica es crear un codigo flexible de entrada.

Una solución para tener un código mas flexible sería **elegir las filas por lo que son, no por dónde están**:

`soup.select("table#example tbody tr")` devuelve solo las filas del cuerpo, haya pie o no.

### 1.3 Alternativa usando `select` y una lista de diccionarios

Veamos otra alternativa para extraer la misma tabla que en la celda anterior. Hagamos dos cambios:

- Ahora probemos utilizar **`select` con un selector CSS** en lugar de `find_all`. Los dos métodos son válidos. Con `find_all` también se puede: `soup.find("tbody").find_all("tr")`. `select` tiene la ventaja de que usa los mismos selectores que ven con **Inspeccionar** en el navegador.
- **Juntar una lista de diccionarios y crear el DataFrame una sola vez**, en lugar de agregar filas con `df.loc[i] = ...` dentro del `for`. Para una tabla chica da lo mismo; con miles de filas, la lista es mucho más rápida, y deja explícito qué columna es cada dato.

`get_text(strip=True)` saca los espacios y saltos de línea que suele haber alrededor del texto.

In [ ]:
def limpia_filas_con_select(html):
    soup = BeautifulSoup(html, "html.parser")
    filas = []
    for tr in soup.select("table#example tbody tr"):        # solo las filas del cuerpo
        celdas = [td.get_text(strip=True) for td in tr.find_all("td")]
        filas.append(dict(zip(["name", "position", "office", "age", "start", "salary"], celdas)))
    return pd.DataFrame(filas)                              # el DataFrame se crea una vez


print(limpia_filas_con_select(html_con_pie).shape, limpia_filas_con_select(html_sin_pie).shape)
limpia_filas_con_select(html_sin_pie)

### 1.4 Tema nuevo: `pd.read_html`

Cuando el dato está en una etiqueta `<table>` bien armada, pandas lo extrae en una línea con `pd.read_html`, que devuelve **una lista** de DataFrames (uno por cada tabla de la página). Necesita `lxml` instalado.

El HTML se le pasa envuelto en `StringIO`. Pasarle el texto directamente quedó deprecado en pandas 2.1, y en pandas 3 da error (lo toma como el nombre de un archivo). Si encuentran en internet `pd.read_html(texto)`, ese es el cambio.

In [ ]:
from io import StringIO

tablas = pd.read_html(StringIO(html_sin_pie))
print(len(tablas), "tabla(s)")
empleados = tablas[0]
empleados.dtypes

**Ejercicio.** `read_html` convirtió `Age` en número, pero `Salary` quedó como texto por el `$` y las comas. Conviértanla en número.

In [ ]:
# Su código acá: conviertan empleados["Salary"] en número


<details><summary><b>Ver una solución</b> (ábranla después de intentarlo)</summary>

```python
empleados["Salary"] = pd.to_numeric(empleados["Salary"].str.replace("$", "", regex=False)
                                              .str.replace(",", "", regex=False))
empleados
```

**Por qué `regex=False`.** Desde pandas 2.0, `.str.replace` trata el patrón como texto literal por defecto; antes el valor por defecto era `regex=True`. Con `regex=True`, `"$"` significa «fin del texto» y no reemplaza nada, y `"."` significa «cualquier carácter» (`"3.219.016"` queda vacío). Mucho código de foros y de IA depende del valor por defecto: escriban siempre `regex=False` (o `True`) explícito.
</details>

Si no les salió, copien la solución en la celda de arriba y ejecútenla antes de seguir: las celdas siguientes la usan.

### 1.5 Tema nuevo: el dato puede estar en un atributo

En muchas páginas el texto que se ve está recortado, y el dato completo está en un **atributo** de la etiqueta. Es muy común en listados de productos. Este fragmento tiene el formato de [books.toscrape.com](https://books.toscrape.com), un sitio hecho para practicar *scraping*.

In [ ]:
html_producto = """
<article class="product_pod">
  <p class="star-rating Three"></p>
  <h3><a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">A Light in the ...</a></h3>
  <p class="price_color">£51.77</p>
</article>"""
producto = BeautifulSoup(html_producto, "html.parser")
enlace = producto.select_one("h3 a")
print("texto visible:", enlace.get_text())
print("atributo title:", enlace["title"])
print("atributo href: ", enlace.get("href"))
print("clases:        ", producto.select_one("p.star-rating")["class"])

- `select_one` devuelve el primer elemento que coincide (o `None`); `select`, una lista con todos.
- `enlace["title"]` da error si el atributo no existe; `enlace.get("title")` devuelve `None`.
- La calificación está en la **clase** (`star-rating Three`). `["class"]` devuelve una lista porque un elemento puede tener varias clases.

### 1.6 Corre, pero puede fallar en silencio: agreguemos precauciones con `timeout`, `raise_for_status()`, `headers`, `write_bytes`

Revisen la slide 16 del video y agreguen estas precauciones para evitar errores silenciosos:

- **`timeout=15`**: si el servidor no contesta, el notebook queda "ejecutando" esa celda **para siempre** sin dar error. Para evitarlo, agreguemos `timeout=15`. De esta forma le damos 15 segundos para que espere el resultado y sino
- **`raise_for_status()`**: si la respuesta es un error (404, 500), lo convierte en un error de Python. Sin esta línea, una página de error se procesaría como si fueran datos.
- **`headers` con un `User-Agent`**: dice quiénes somos. Algunos sitios rechazan los pedidos que no lo tienen.
- **Guardar el HTML crudo** antes de procesarlo: si la página cambia o desaparece, el análisis se puede repetir.

In [ ]:
# [necesita internet]
from pathlib import Path
import requests

URL_DT = "https://datatables.net/examples/basic_init/zero_configuration.html"
ENCABEZADOS = {"User-Agent": "Curso Python para Economistas (UNLP) - uso educativo"}

respuesta = requests.get(URL_DT, headers=ENCABEZADOS, timeout=15)
respuesta.raise_for_status()                                    # 4xx o 5xx: error ruidoso
print(respuesta.status_code, respuesta.headers.get("Content-Type"))

Path("archivos/crudo").mkdir(parents=True, exist_ok=True)
Path("archivos/crudo/datatables.html").write_bytes(respuesta.content)   # el HTML tal cual llegó

soup = BeautifulSoup(respuesta.content, "html.parser")
filas = soup.select("table#example tbody tr")
print(len(filas), "filas en el HTML")

**Lo que deberían ver:** un código `200` y **57 filas**, no 10. Confirma la respuesta de 1.1.

Tres definiciones para tener presentes:

- **Código de estado**: 200 es OK; 3xx, una redirección; 4xx, un error nuestro (404 no existe, 403 prohibido, 429 demasiados pedidos); 5xx, un error del servidor. **Un 200 dice que el servidor respondió, no que los datos sean correctos ni que estén completos.**
- **`respuesta.content`** son los bytes tal cual llegaron; **`respuesta.text`** es ese contenido decodificado con la codificación que *adivina* `requests`. Si el servidor no la declara, `requests` asume ISO-8859-1 para el HTML, y aparecen cosas como `Â£`. Pasarle `content` a BeautifulSoup deja que use la codificación declarada dentro del propio HTML.
- **Ética y legalidad.** Antes de *scrapear*: miren el archivo `robots.txt` del sitio y sus términos de uso; si hay una API o una descarga oficial, úsenla; no hagan cientos de pedidos por segundo (`time.sleep` entre pedidos); guarden el HTML crudo para no pedirlo de nuevo; y cuidado con los datos personales (en Argentina, Ley 25.326).

## 2. Descargar un archivo

La slide 9 descargamos un archivo con `requests` y lo guardamos con `open(nombre, "w")`, `.decode()` y `close()`.

**Otra opción:** Si lo que quieren es un CSV para analizarlo, `pd.read_csv` acepta directamente una URL: `pd.read_csv(url)`. Sin embargo, esta opcion no nos guarda una copia del archivo; para un trabajo reproducible conviene guardarlo primero.

**Otra opción más precavida:** En la slide 9, usamos `open(nombre, "w")` para abrir y pasar el texto a bytes, con la codificación por defecto de su computadora (en Windows, casi nunca es UTF-8). Según qué caracteres tenga el archivo, esto puede dar error; y si algo falla antes del `close()`, el archivo queda abierto.

Entonces, una practica más confiable es guardar los **bytes** tal cual llegaron y recien a continuación decodificarlos y escribirlos como texto:

```python
respuesta = requests.get(url, timeout=30)
respuesta.raise_for_status()
Path("datos/archivo.csv").write_bytes(respuesta.content)
```



## 3. Tema nuevo: APIs con código

En el video, la API era solo una definición (slide 29). Acá la usamos.

Una **API web** es una dirección a la que un programa le pide datos con una URL y recibe una respuesta estructurada, casi siempre en **JSON**. Es la mejor fuente posible: los datos vienen pensados para ser leídos por un programa.

Vamos a usar la **API Georef** del Estado argentino, que normaliza nombres geográficos y trae los códigos oficiales del INDEC. Su documentación está en [argentina.gob.ar/georef](https://www.argentina.gob.ar/georef). La versión actual es la **2.1**. Una consulta tiene tres partes:

```
https://apis.datos.gob.ar/georef/api/v2.0/departamentos?provincia=06&max=50
└──────────── URL base ────────────────┘└── recurso ──┘└── parámetros ──┘
```

Definiciones:

- **Recurso** (o *endpoint*): cada dirección de la API (`/provincias`, `/departamentos`, `/direcciones`).
- **Parámetros**: lo que va después del `?`, como pares `clave=valor` separados por `&`.
- **JSON**: texto con objetos `{...}` y listas `[...]`. `respuesta.json()` lo convierte en diccionarios y listas de Python.
- **Paginación**: la API devuelve los resultados de a partes. En Georef, `max` dice cuántos traer (10 si no se dice nada) e `inicio`, desde cuál.
- **Autenticación**: muchas APIs piden una clave (*API key*); Georef no.
  - **ATENCION:** Nunca suban una clave a GitHub.


### 3.1 Una función para consultar

En lugar de armar la URL a mano con un *f-string* (`f"...?provincia={p}&max={n}"`), se le pasan los parámetros a `requests` en un diccionario con `params`. `requests` se ocupa de codificar las tildes, los espacios y los `&`, y la URL que realmente pidió queda en `respuesta.url`.

In [ ]:
import json
import time
from datetime import date
from pathlib import Path

import requests

BASE_GEOREF = "https://apis.datos.gob.ar/georef/api/v2.0"


def consultar(recurso, **params):
    """GET a la API Georef. Devuelve el JSON como diccionario.

    recurso: "provincias", "departamentos", "municipios", ...
    params:  parámetros de la consulta, por ejemplo provincia="06", max=50.
    """
    respuesta = requests.get(f"{BASE_GEOREF}/{recurso}", params=params, timeout=15)
    print(respuesta.status_code, respuesta.url)        # la URL que realmente se pidió
    respuesta.raise_for_status()
    return respuesta.json()

### 3.2 `cantidad` no es lo mismo que `total`

**Predecir.** Pedimos las provincias **sin ningún parámetro**. ¿Cuántas vienen?

In [ ]:
# [necesita internet]
datos = consultar("provincias")
print(type(datos), list(datos.keys()))
print("cantidad:", datos["cantidad"], "· total:", datos["total"], "· inicio:", datos["inicio"])

<details><summary><b>Ver la respuesta</b></summary>

Vienen **10 de 24**. La respuesta es un diccionario con las claves `cantidad` (lo que vino), `total` (lo que existe), `inicio`, `parametros` y `provincias`. Georef devuelve 10 resultados si no se le pide otra cantidad.

Este es **el error silencioso más común con APIs**: el código corre, el DataFrame se ve bien, el código de estado es 200, pero faltan datos! Siempre comparen lo que vino con el total que informa la API.
</details>

Lo arreglamos con `max`. Para pasar la lista a un DataFrame usamos `pd.json_normalize`, que además **aplana** los diccionarios anidados: `centroide` se convierte en las columnas `centroide.lat` y `centroide.lon`.

In [ ]:
# [necesita internet]
datos = consultar("provincias", max=30)
assert datos["cantidad"] == datos["total"], "Faltan resultados: pidan más con max o paginen"

provincias_api = pd.json_normalize(datos["provincias"])
print(provincias_api.dtypes)
provincias_api.sort_values("id").head()

**Lo que deberían ver:** 24 filas y las columnas `id`, `nombre`, `centroide.lat` y `centroide.lon`. Fíjense que **el `id` viene como texto** (`"02"`, `"06"`): en el JSON está entre comillas, y el cero de adelante se conserva. Es el código del INDEC, la clave que vamos a usar para unir tablas en la sección 5 y para los mapas de la clase 6.

### 3.3 Ejercicio: traer todas las páginas

**Predecir.** Pedimos los departamentos (partidos) de la provincia de Buenos Aires, cuyo código es `06`, con `max=50`. ¿Cuántos vienen y cuántos hay?

In [ ]:
# [necesita internet]
datos = consultar("departamentos", provincia="06", max=50)
print("cantidad:", datos["cantidad"], "· total:", datos["total"])

Vienen 50 y hay más. Muchas APIs ponen un máximo por pedido: el Banco Mundial, por ejemplo, devuelve 50 por página. La solución general es **paginar**: pedir de a `paso` resultados, mover `inicio` y parar cuando se juntó el `total`.

**Ejercicio.** Completen la función. Entre pedido y pedido, `time.sleep(0.2)` para no saturar el servidor.

In [ ]:
def traer_todos(recurso, paso=50, **params):
    """Pide todas las páginas de un recurso de Georef. Devuelve (lista de resultados, total informado)."""
    resultados = []
    inicio = 0
    while True:
        # 1. consultar(recurso, max=..., inicio=..., **params)
        # 2. agregar los resultados de esta página a la lista (¿bajo qué clave vienen?)
        # 3. mover inicio
        # 4. si ya se juntó el total, devolver (resultados, total)
        # 5. time.sleep(0.2)
        ...


<details><summary><b>Ver una solución</b> (ábranla después de intentarlo)</summary>

```python
def traer_todos(recurso, paso=50, **params):
    """Pide todas las páginas de un recurso de Georef. Devuelve (lista de resultados, total informado)."""
    resultados = []
    inicio = 0
    while True:
        datos = consultar(recurso, max=paso, inicio=inicio, **params)
        resultados.extend(datos[recurso])        # la lista viene bajo la clave con el nombre del recurso
        inicio += paso
        if inicio >= datos["total"] or datos["cantidad"] == 0:
            return resultados, datos["total"]
        time.sleep(0.2)                          # cortesía con el servidor
```

Errores frecuentes: usar `append` en lugar de `extend` (queda una lista de listas); no mover `inicio` (el bucle no termina nunca, por eso conviene también la condición `cantidad == 0`); o moverlo de a 1 en lugar de `paso` (salen repetidos).
</details>

Si no les salió, copien la solución en la celda de arriba y ejecútenla antes de seguir: las celdas siguientes la usan.

Ejecuten la verificación:

In [ ]:
# [necesita internet]
deptos, total = traer_todos("departamentos", provincia="06")
deptos_ba = pd.json_normalize(deptos)

# Verificación
assert len(deptos_ba) == total, f"Vinieron {len(deptos_ba)} de {total}"
assert deptos_ba["id"].is_unique, "Hay departamentos repetidos: revisen cómo mueven inicio"
assert (deptos_ba["provincia.id"] == "06").all(), "Vinieron departamentos de otra provincia"
print("✔", len(deptos_ba), "departamentos de", deptos_ba["provincia.nombre"].iloc[0])

**Para pensar:** prueben `traer_todos("departamentos", provincia="buenos aires")`, con el nombre en lugar del código. Si la búsqueda por texto trae también las comunas de la Ciudad Autónoma de Buenos Aires, el tercer `assert` lo detecta. Es un buen ejemplo de por qué conviene filtrar por código y no por nombre.

### 3.4 Errores: ¿cuáles de estos pedidos fallan?

**Predecir.** Ejecuten la celda y comparen. ¿Por qué el primero **no** es un error para la API?

In [ ]:
# [necesita internet]
casos = {
    "nombre que no existe": ("provincias", {"nombre": "Narnia"}),
    "parámetro inválido":   ("provincias", {"max": "mil"}),
    "recurso mal escrito":  ("provinciass", {}),
}
for descripcion, (recurso, params) in casos.items():
    try:
        datos = consultar(recurso, **params)
        print(f"  {descripcion}: sin error, cantidad = {datos['cantidad']}")
    except requests.HTTPError as error:
        print(f"  {descripcion}: HTTPError {error.response.status_code}")
    except requests.RequestException as error:        # sin conexión, timeout, etc.
        print(f"  {descripcion}: falló la conexión ({type(error).__name__})")

<details><summary><b>Ver la respuesta</b></summary>

Lo esperable: «Narnia» devuelve 200 con `cantidad = 0`. Una búsqueda sin resultados es una respuesta válida, así que **hay que chequear que no venga vacía**. `max="mil"` devuelve 400, un error nuestro: el JSON trae una clave `errores` con el motivo. `provinciass` devuelve 404.

`requests.RequestException` agrupa todos los errores de `requests`: falta de conexión, `Timeout`, `HTTPError`. Un `except:` sin nada también atraparía los errores de nuestro propio código (por ejemplo, un `KeyError`) y los escondería.
</details>

### 3.5 Guardar la respuesta cruda

Una API puede cambiar o dejar de estar disponible. Guardar el JSON tal como llegó, con la fecha en el nombre, permite repetir el análisis.

In [ ]:
# [necesita internet]
def guardar_crudo(datos, nombre, carpeta="archivos/crudo"):
    ruta = Path(carpeta) / f"{nombre}_{date.today():%Y%m%d}.json"
    ruta.parent.mkdir(parents=True, exist_ok=True)
    ruta.write_text(json.dumps(datos, ensure_ascii=False, indent=2), encoding="utf-8")
    return ruta


print(guardar_crudo({"departamentos": deptos, "total": total}, "georef_departamentos_06"))

## 4. PDF

### 4.1 ⛔ Reemplazo obligatorio: `PyPDF2` → `pypdf`

`PyPDF2` está **discontinuado**: el proyecto siguió con el nombre `pypdf`. No hace falta desinstalar `PyPDF2`, simplemente instalen `pypdf` y usen ese. La lógica es idéntica: abrir el PDF, recorrer las páginas, sacar el texto. Solo cambiaron los nombres.

**Si en [stack overflow](https://stackoverflow.com/) o alguna IA les sugiere `PdfFileReader` o `extractText`, ya saben que el código es viejo y que conviene reemplazarlo.** Lo que tienen que saber es reconocer el código viejo, porque está en todos lados:

| En la slide 20 | En el Jupyter |
| --- | --- |
| `import PyPDF2` | `from pypdf import PdfReader` |
| `PyPDF2.PdfFileReader(archivo)` | `PdfReader("boletin_ife.pdf")` |
| `pdfReader.numPages` | `len(lector.pages)` |
| `pdfReader.getPage(i)` | `lector.pages[i]` |
| `pagina.extractText()` | `pagina.extract_text()` |

```python
from pypdf import PdfReader
lector = PdfReader("boletin_ife.pdf")
for pagina in lector.pages:
    print(pagina.extract_text())
```



Por otro lado, **Minecart** (slide 5) casi no tiene mantenimiento, asi que una opcion más nueva con la que podemos reemplazarlo es `pdfplumber` da acceso a lo mismo (caracteres, líneas, rectángulos, imágenes).

In [ ]:
import re
import pdfplumber
from pypdf import PdfReader

RUTA_PDF = "boletin_ife.pdf"
lector = PdfReader(RUTA_PDF)
print(len(lector.pages), "páginas")
print(lector.pages[6].extract_text()[:300])        # el principio de la página 7

### 4.2 Veamos una alternativa a `tabula` por si no tienen Java

`tabula` (slide 21) necesita **Java** instalado. Si su computadora no lo tiene pueden usar `pdfplumber` que hace lo mismo y es solo Python. Pueden usar la que prefieran, con dos advertencias:

1. **Las dos pueden extraer mal la misma tabla.** Ninguna herramienta los exime de verificar (sección 4.3).
2. **En la versión actual de `tabula-py`** **bold text** Si le pasan `output_format="dataframe"` explícitamente, `tabula` ignora `multiple_tables=True` y arma la tabla como un único CSV: según la tabla, da un error (`CSVParseError`) o, peor, corre y lee mal los números. Y si no le dicen que no hay encabezado, usa la primera fila de datos como nombres de columnas.

Así queda la tabla de la página 7 con `pdfplumber`, y abajo, la versión con `tabula` corregida. La de `tabula` está dentro de un `try`: si no tienen Java, imprime un aviso y pueden seguir.

In [ ]:
with pdfplumber.open(RUTA_PDF) as pdf:
    tabla = pdf.pages[6].extract_tables()[0]          # página 7: lista de filas, cada fila una lista

tabla_p7 = pd.DataFrame(tabla[1:], columns=["tipo_de_hogar", "pobreza", "indigencia"])
for col in ["pobreza", "indigencia"]:
    tabla_p7[col] = pd.to_numeric(tabla_p7[col].str.replace("%", "", regex=False)
                                               .str.replace(",", ".", regex=False))
tabla_p7

In [ ]:
try:
    import tabula

    lista = tabula.read_pdf(RUTA_PDF, pages=7, pandas_options={"header": None, "dtype": str})
    tabla_p7_tabula = lista[0]
    titulos = tabla_p7_tabula.iloc[0].fillna("tipo_de_hogar").str.lower()     # la primera fila trae los títulos
    tabla_p7_tabula = (tabla_p7_tabula.iloc[1:].set_axis(titulos, axis=1)
                       .reset_index(drop=True).rename_axis(columns=None))
    print(tabla_p7_tabula)
except Exception as error:
    print("tabula no funcionó en este entorno:", type(error).__name__, "→ sigan con pdfplumber o pypdf")

Si `tabula.read_pdf` les da un error raro en Colab (`CSVParseError` o `JSONDecodeError`), prueben `tabula.convert_into(...)`, que hace que Java escriba la tabla directamente en un CSV, y léanlo con `pd.read_csv(..., header=None, dtype=str)`. O usen `pdfplumber` o `pypdf`.

### 4.3 Tema nuevo: el desafío del Cuadro 5

La página 7 es fácil. La mayoría de las tablas de los informes no lo son. Esta parte es la más importante del notebook: es lo más parecido a lo que van a hacer en su tesis.

El Ingreso Familiar de Emergencia (IFE) fue la transferencia que ANSES pagó en 2020. El **Cuadro 5** del boletín (página 27) tiene, por provincia, la población, cuántas personas lo solicitaron, a cuántas se les liquidó y a cuántas se rechazó.

**El objetivo:** un DataFrame `cuadro5` con una fila por provincia (más «Sin información» y «Total») y **números de verdad**:

| Columna | Tipo | Ejemplo (Buenos Aires) |
| --- | --- | --- |
| `provincia` | texto | `Buenos Aires` |
| `poblacion`, `solicitantes`, `liquidados`, `rechazados` | entero (`Int64`) | `17541141`, `4921666`, `3219016`, `1702650` |
| `liq_pob`, `sol_pob`, `rech_sol` | número (porcentaje) | `18.4`, `28.1`, `34.6` |

**Y la condición más importante: que puedan demostrar que está bien.** El cuadro trae una fila de totales y porcentajes calculados: si la extracción fue correcta, todo tiene que cerrar.

El método, que les va a servir para cualquier PDF: **mirar, extraer, detectar el problema, probar otra estrategia, convertir y verificar.** Abran el PDF en la página 27 y **mírenlo antes de programar**.

**Paso 1. Encontrar la página.** En lugar de escribir el número de página de memoria, búsquenlo.

**Predecir:** ¿en cuántas páginas aparece el texto `"Cuadro 5"`?

In [ ]:
con_cuadro5 = [i + 1 for i, pagina in enumerate(lector.pages) if "Cuadro 5" in pagina.extract_text()]
print("Páginas con 'Cuadro 5':", con_cuadro5)

Aparece en cuatro páginas: en tres, el texto dice «Fuente: ... a partir del Cuadro 5 del Anexo». **Ejercicio:** ajusten la búsqueda para que encuentre solo la página del cuadro.

In [ ]:
# Su código acá


<details><summary><b>Ver una solución</b> (ábranla después de intentarlo)</summary>

```python
pagina_cuadro5 = [i + 1 for i, pagina in enumerate(lector.pages) if "Cuadro 5." in pagina.extract_text()]
print(pagina_cuadro5)
```

Con `"Cuadro 5."`, con el punto del título, queda solo la 27. Una búsqueda que devuelve algo no es una búsqueda que devuelve lo correcto.
</details>

Si no les salió, copien la solución en la celda de arriba y ejecútenla antes de seguir: las celdas siguientes la usan.

**Paso 2. Primer intento: `extract_tables()`.**

**Predecir:** ¿cuántas filas y columnas va a tener la tabla?

In [ ]:
with pdfplumber.open(RUTA_PDF) as pdf:
    pagina = pdf.pages[26]
    tablas = pagina.extract_tables()
    texto_plumber = pagina.extract_text()

print(len(tablas), "tabla(s);", [f"{len(t)} filas x {len(t[0])} columna(s)" for t in tablas])
print(repr(tablas[0][1][0][:160]))

Todo el cuadro quedó en **una sola celda**, con las filas separadas por `\n`. `extract_tables()` busca las líneas del dibujo de la tabla, y este cuadro (pegado desde Excel a Word) no las tiene donde las espera. Plan B: el texto de la página, línea por línea.

**Paso 3. Segundo intento: el texto y una expresión regular.** Cada fila del cuadro es una línea: un nombre, cuatro números y tres porcentajes. Escriban `PATRON_FILA` con **grupos con nombre** (clase 4), uno por columna, anclado al principio y al final de la línea (`^...$`). Les dejamos los patrones de un número y de un porcentaje.

Cuidado con la fila «Sin información»: no tiene población (dice `-`) ni los dos primeros porcentajes. Pista: un pedazo opcional del patrón se escribe `(?: ... )?`.

In [ ]:
NUM = r"\d{1,3}(?:\.\d{3})*"     # 805 · 34.857 · 3.219.016  (puntos de miles)
PCT = r"\d{1,3},\d%"             # 18,4%                     (coma decimal)

print(re.fullmatch(NUM, "3.219.016"), re.fullmatch(NUM, "3219016"), re.fullmatch(NUM, "4 .921.666"))
print(re.fullmatch(PCT, "18,4%"))

In [ ]:
PATRON_FILA = re.compile(
    rf"^(?P<provincia>[^\d]+?)\s+"
    # completen: solicitantes, liquidados, rechazados y los tres porcentajes
    # (los dos primeros porcentajes pueden faltar)

)

<details><summary><b>Ver una solución</b> (ábranla después de intentarlo)</summary>

```python
PATRON_FILA = re.compile(
    rf"^(?P<provincia>[^\d]+?)\s+"
    rf"(?P<poblacion>{NUM}|-)\s+(?P<solicitantes>{NUM})\s+(?P<liquidados>{NUM})\s+(?P<rechazados>{NUM})\s+"
    rf"(?:(?P<liq_pob>{PCT})\s+(?P<sol_pob>{PCT})\s+)?(?P<rech_sol>{PCT})$"
)
```

Si el nombre de la provincia lo escribieron con `.*`, el cuantificador codicioso se come números; usen `[^\d]+?` o el perezoso `.+?` (clase 4). Y sin `^` y `$`, el patrón «encuentra algo» en filas rotas, que quedan mal partidas sin aviso.
</details>

Si no les salió, copien la solución en la celda de arriba y ejecútenla antes de seguir: las celdas siguientes la usan.

**Punto de control.** Prueben el patrón con una línea escrita a mano, y después cuenten cuántas líneas del texto de `pdfplumber` coinciden.

In [ ]:
linea_ejemplo = "Buenos Aires 17.541.141 4.921.666    3.219.016 1.702.650   18,4% 28,1% 34,6%"
m = PATRON_FILA.match(linea_ejemplo)
print(m.groupdict() if m else "El patrón no coincide con la línea de ejemplo: revísenlo antes de seguir")

lineas = texto_plumber.splitlines()
con_numeros = [l for l in lineas if re.search(r"\d\.\d{3}", l)]
print(len(con_numeros), "líneas con números ·", sum(bool(PATRON_FILA.match(l.strip())) for l in con_numeros), "coinciden")

**Predecir antes de mirar:** si el patrón funciona con la línea de ejemplo, ¿por qué no coincide con **ninguna** de las del PDF? Impriman tres líneas y mírenlas con lupa.

In [ ]:
for linea in con_numeros[:3]:
    print(repr(linea))

Si su patrón coincide con la línea de ejemplo y con 0 líneas del PDF, **está bien**: el patrón está bien, el texto está mal. Si coincidiera con algunas, sería mala señal: probablemente le falta `^` o `$`, y esas filas quedaron partidas mal.

**Paso 4. Diagnóstico.** Aparece `4 .921.666` en lugar de `4.921.666`. ¿De dónde sale ese espacio? Miremos los **caracteres** que `pdfplumber` encontró en esa zona de la página, con su posición horizontal (`x0` y `x1`, en puntos desde el borde izquierdo).

In [ ]:
palabra = next(w for w in pagina.extract_words() if w["text"] == "Aires")       # la línea de Buenos Aires
misma_linea = [c for c in pagina.chars if abs(c["top"] - palabra["top"]) < 2]
zona = sorted((c for c in misma_linea if 220 < c["x0"] < 245), key=lambda c: c["x0"])   # donde empieza 4.921.666
for c in zona:
    print(f"{c['text']!r:5} desde x = {c['x0']:6.1f} hasta x = {c['x1']:6.1f}")

Hay un **espacio** que empieza exactamente donde empieza el `4`. El cuadro se armó en Excel con los números alineados a la derecha y rellenados con espacios, y al pegarlo en Word esos espacios quedaron **encimados** con los dígitos. `pdfplumber` ordena los caracteres por su posición en la página, y el espacio cae en el medio del número. `pypdf`, en cambio, lee los caracteres en el orden en que están guardados en el archivo.

Hay tres arreglos. Elijan uno; si les sobra tiempo, prueben un segundo y comparen:

- **Camino A, `pypdf`:** `lector.pages[26].extract_text()`.
- **Camino B, `pdfplumber` sin los espacios encimados:** filtrar los caracteres de espacio y dejar que `pdfplumber` decida dónde van, con una tolerancia horizontal chica: `pagina.filter(lambda obj: not (obj["object_type"] == "char" and obj["text"] == " ")).extract_text(x_tolerance=1.5)`.
- **Camino C, `tabula`** (requiere Java): `tabula.read_pdf(RUTA_PDF, pages=27)[0]`. Devuelve un DataFrame, pero el encabezado de dos renglones queda partido y los números siguen siendo texto.

**Ejercicio.** Escriban `extraer_filas(texto)`, que devuelva un DataFrame **de textos** con una fila por cada línea que coincide con `PATRON_FILA`.

In [ ]:
def extraer_filas(texto):
    """Una fila del DataFrame por cada línea de texto que coincide con PATRON_FILA (todo como texto)."""
    ...


texto_a = ...          # el texto de la página con el camino que eligieron
filas_a = extraer_filas(texto_a)
filas_a.tail(3)

<details><summary><b>Ver una solución</b> (ábranla después de intentarlo)</summary>

```python
def extraer_filas(texto):
    filas = [m.groupdict() for linea in texto.splitlines() if (m := PATRON_FILA.match(linea.strip()))]
    return pd.DataFrame(filas)


texto_a = lector.pages[26].extract_text()                                   # camino A
with pdfplumber.open(RUTA_PDF) as pdf:                                      # camino B
    sin_espacios = pdf.pages[26].filter(lambda obj: not (obj["object_type"] == "char" and obj["text"] == " "))
    texto_b = sin_espacios.extract_text(x_tolerance=1.5)

filas_a = extraer_filas(texto_a)
filas_b = extraer_filas(texto_b)
print(filas_a.shape, filas_b.shape, "· iguales:", filas_a.equals(filas_b))
filas_a.tail(3)
```

Los dos caminos dan 26 filas (24 provincias, «Sin información» y «Total») y son idénticos. Eso es una **verificación cruzada**: dos métodos independientes que coinciden. En el camino B, con `x_tolerance=2` ya vuelven a pegarse palabras («Santiagodel Estero»): el ajuste es frágil. Lo que da confianza no es el truco, sino la verificación del paso 6.
</details>

Si no les salió, copien la solución en la celda de arriba y ejecútenla antes de seguir: las celdas siguientes la usan.

In [ ]:
# Punto de control
assert len(filas_a) > 0, "Ninguna línea coincidió: ¿usaron el texto del camino A o B, o el de pdfplumber sin arreglar?"
assert len(filas_a) == 26, f"Esperábamos 26 filas y hay {len(filas_a)}: comparen las provincias con las del PDF"
print("✔", len(filas_a), "filas extraídas")

**Paso 5. De texto a números.** Escriban dos funciones que reciban una **Serie** de textos:

- `a_numero`: `"3.219.016"` → `3219016`, y `"-"` → faltante. Devuelve tipo `Int64` (con mayúscula: el entero de pandas que admite faltantes; el `int64` común no los admite).
- `a_porcentaje`: `"18,4%"` → `18.4`. Los porcentajes que faltan quedan como faltante.

Después armen `cuadro5`. **No usen `errors="coerce"`** en `pd.to_numeric`: convierte cualquier problema en un faltante sin avisar, y es justo lo que no queremos.

In [ ]:
def a_numero(serie):
    """'3.219.016' -> 3219016 ; '-' -> <NA>"""
    ...


def a_porcentaje(serie):
    """'18,4%' -> 18.4"""
    ...


cuadro5 = filas_a.copy()
# conviertan las columnas
cuadro5.head()

<details><summary><b>Ver una solución</b> (ábranla después de intentarlo)</summary>

```python
def a_numero(serie):
    """'3.219.016' -> 3219016 ; '-' -> <NA>"""
    limpia = serie.replace("-", pd.NA).str.replace(".", "", regex=False)
    return pd.to_numeric(limpia).astype("Int64")


def a_porcentaje(serie):
    """'18,4%' -> 18.4"""
    limpia = serie.str.replace("%", "", regex=False).str.replace(",", ".", regex=False)
    return pd.to_numeric(limpia)


cuadro5 = filas_a.copy()
for columna in ["poblacion", "solicitantes", "liquidados", "rechazados"]:
    cuadro5[columna] = a_numero(cuadro5[columna])
for columna in ["liq_pob", "sol_pob", "rech_sol"]:
    cuadro5[columna] = a_porcentaje(cuadro5[columna])
print(cuadro5.dtypes)
cuadro5.head()
```

</details>

Si no les salió, copien la solución en la celda de arriba y ejecútenla antes de seguir: las celdas siguientes la usan.

**Paso 6. Verificar.** Antes de ejecutar la celda siguiente, escriban: ¿qué relaciones tienen que cumplirse en este cuadro si la extracción es correcta? Anoten al menos tres.

<details><summary><b>Ver la respuesta</b></summary>

Las cuatro que verifica la celda de abajo:

1. Hay 24 provincias, más «Sin información» y «Total».
2. La suma de las filas, **incluida «Sin información»**, es igual al Total, columna por columna. Si dejan afuera esa fila, no cierra por 34.857 solicitantes. Es decir, que el total la incluye.
3. En cada fila, solicitantes = liquidados + rechazados.
4. Los porcentajes recalculados coinciden con los publicados, con una tolerancia de redondeo (el PDF los redondea a un decimal; con `==` fallaría).

Dos palabras para tener presentes: **verificación interna** es usar las relaciones que el propio documento garantiza (sumas, totales, porcentajes); **verificación cruzada** es comparar dos métodos o dos fuentes.
</details>

In [ ]:
# Verificación del desafío: no la modifiquen
prov = cuadro5[~cuadro5["provincia"].isin(["Total"])]
total = cuadro5.set_index("provincia").loc["Total"]

assert len(cuadro5) == 26, f"Esperábamos 26 filas (24 provincias + Sin información + Total) y hay {len(cuadro5)}"
assert {"Buenos Aires", "CABA", "Tierra del Fuego", "Sin información", "Total"} <= set(cuadro5["provincia"]), \
    "Revisen los nombres: ¿quedaron espacios de más?"
for col in ["poblacion", "solicitantes", "liquidados", "rechazados"]:
    assert str(cuadro5[col].dtype) == "Int64", f"{col} debería ser Int64 y es {cuadro5[col].dtype}"
    assert prov[col].sum() == total[col], f"{col}: las filas suman {prov[col].sum():,} y el total dice {total[col]:,}"
assert (cuadro5["solicitantes"] == cuadro5["liquidados"] + cuadro5["rechazados"]).all(), \
    "En alguna fila solicitantes != liquidados + rechazados"
for pct, num, den in [("liq_pob", "liquidados", "poblacion"), ("sol_pob", "solicitantes", "poblacion"),
                      ("rech_sol", "rechazados", "solicitantes")]:
    diferencia = ((cuadro5[num] / cuadro5[den] * 100).astype(float) - cuadro5[pct]).abs()
    assert diferencia.notna().sum() >= 25, f"{pct}: hay demasiados faltantes"
    assert (diferencia.dropna() < 0.051).all(), f"{pct} no coincide con {num}/{den} (diferencia máxima {diferencia.max():.2f})"
print("✔ El Cuadro 5 está completo y cierra: sumas, identidades y porcentajes")

**Paso 7. Interpretar.** Con `cuadro5` verificado:

(a) ¿Qué cinco provincias tienen mayor proporción de liquidados sobre su población? ¿Y menor?

(b) ¿Dónde se rechazó una proporción mayor de las solicitudes? Propongan una hipótesis económica: ¿qué tienen en común esas provincias? (El Cuadro 13, en la página 33, lista los motivos de rechazo; el primero es «Relación de dependencia».)

(c) La columna `poblacion`, ¿es del censo? Busquen en el PDF la fuente del cuadro.

In [ ]:
# Su código acá


<details><summary><b>Ver una solución</b> (ábranla después de intentarlo)</summary>

```python
solo_provincias = cuadro5[~cuadro5["provincia"].isin(["Total", "Sin información"])]
print(solo_provincias.nlargest(5, "liq_pob")[["provincia", "liq_pob"]].to_string(index=False))
print()
print(solo_provincias.nsmallest(5, "liq_pob")[["provincia", "liq_pob"]].to_string(index=False))
print()
print(solo_provincias.nlargest(5, "rech_sol")[["provincia", "rech_sol"]].to_string(index=False))
```

</details>

Si no les salió, copien la solución en la celda de arriba y ejecútenla antes de seguir: las celdas siguientes la usan.

<details><summary><b>Ver la respuesta</b></summary>

(a) Mayor cobertura: Chaco (28,9 %), Santiago del Estero (28,6 %), Formosa (27,1 %), Corrientes (26,7 %) y Salta (25,5 %), en el NEA y el NOA. Menor: CABA (11,9 %), Chubut (12,5 %), Santa Cruz (12,6 %), Tierra del Fuego (13,0 %) y Neuquén (15,6 %).

(b) Más rechazos: Tierra del Fuego (48,0 %), CABA (43,7 %), Chubut (41,8 %), Río Negro (40,3 %) y Santa Cruz (38,9 %). Una hipótesis razonable: más empleo formal. Quien lo solicitaba tenía más probabilidad de que alguien de su grupo familiar estuviera en relación de dependencia, el principal motivo de rechazo (1.987.804 de 4.554.262, el 43,6 %). Es una hipótesis: para sostenerla habría que cruzar con datos de empleo. Un cuadro descriptivo no prueba causas.

(c) No. Dice «proyecciones de población del INDEC». Un denominador también es un dato que hay que documentar.
</details>

## 5. SQL con SQLite

Qué es una base relacional? Una tabla relacional en SQL es una estructura de datos organizada en filas (registros o tuplas) y columnas (campos o atributos) que almacena información sobre una entidad específica y se conecta con otras tablas mediante claves primarias y foráneas.

Ya aprendimos sobre SQLite (sin servidor) y PostgreSQL (con servidor) en las slides 23 y 24. Vimos: `SELECT`, `FROM`, `WHERE`, `ORDER BY`, `CREATE`, `DROP`, `INSERT` (slide 25), y desde Python `sqlite3.connect`, `execute`, `executemany` con `?`, `commit` y `close` (slides 26 y 27).

Definiciones que vamos a usar:

- **Clave primaria**: la columna que identifica cada fila sin repetirse (`id` en la tabla de provincias).
- **Clave foránea**: una columna que apunta a la clave primaria de otra tabla.
- **JOIN**: combinar tablas por una clave. `INNER JOIN` deja solo las filas que tienen pareja en las dos tablas; `LEFT JOIN` conserva todas las de la izquierda.

Vamos a guardar dos tablas: `cuadro5` (lo que extrajeron) y `provincias`, con los códigos oficiales. Para que esta parte no dependa de internet, la tabla de provincias sale de `provincias_ign.csv`, con los códigos y nombres de la capa de provincias del Instituto Geográfico Nacional (los mismos que devuelve la API Georef).

### 5.1 Complejicemos lo que vimos en la slide 26

- **Si corremos `CREATE TABLE` funciona la primera vez y pero daría error si ejecutan la celda de nuevo.** Para evitarlo, podemos usar `df.to_sql("tabla", con, if_exists="replace")`, que crea la tabla a partir de un DataFrame y la reemplaza si ya existe. De esta forma, pueden ejecutar esa celda todas las veces que quieran sin enfrentar el error de que la tabla ya exista.
- **Los códigos de provincia combiene guardarlos como strings.** Un código es una **etiqueta**, no una cantidad. Usen los códigos del INDEC como texto (`"02"`, `"06"`). Para ello podemos leer el CSV con `dtype={"id": str}`.

In [ ]:
import sqlite3

provincias = pd.read_csv("provincias_ign.csv", dtype={"id": str})

con = sqlite3.connect("ife.db")                  # crea el archivo si no existe
cuadro5.to_sql("cuadro5", con, if_exists="replace", index=False)
provincias.to_sql("provincias", con, if_exists="replace", index=False)

pd.read_sql_query("SELECT name, sql FROM sqlite_master WHERE type = 'table'", con)

La columna `sql` muestra cómo quedó definida cada tabla: `id TEXT` conserva el `"02"`.

### 5.2 Otro metodo: `pd.read_sql_query`

En el video usamos `c.execute(...)` y `c.fetchall()`, que devuelve una lista de tuplas.
Si después de obtener los datos van a trabajar en pandas existe otra opción que es utilizar el metodo `pd.read_sql_query(consulta, con)`. Este hace la misma consulta y devuelve un **DataFrame** (ahorran el paso de convertirlo en df después)

**Ejercicio.** Las cinco provincias con más liquidados por habitante, escrito en SQL.

In [ ]:
pd.read_sql_query("""
    SELECT ...
    FROM cuadro5
    WHERE ...
    ORDER BY ...
    LIMIT 5
""", con)

<details><summary><b>Ver una solución</b> (ábranla después de intentarlo)</summary>

```python
pd.read_sql_query("""
    SELECT provincia, liquidados, poblacion, liq_pob
    FROM cuadro5
    WHERE provincia NOT IN ('Total', 'Sin información')
    ORDER BY liq_pob DESC
    LIMIT 5
""", con)
```

</details>

Si no les salió, copien la solución en la celda de arriba y ejecútenla antes de seguir: las celdas siguientes la usan.

### 5.3 Tema nuevo: la división entera

**Predecir.** ¿Qué devuelve esta consulta, que calcula la proporción de liquidados en todo el país?

In [ ]:
pd.read_sql_query("""
    SELECT SUM(liquidados) / SUM(poblacion) AS proporcion
    FROM cuadro5
    WHERE provincia NOT IN ('Total', 'Sin información')
""", con)

Da **0**. En SQLite, un entero dividido por un entero es una **división entera** (como `7 // 2` en Python). En pandas, la misma cuenta da decimales: es una trampa clásica al pasar de un lenguaje a otro. Multipliquen por `100.0` (con punto) para que la cuenta se haga con decimales:

In [ ]:
pd.read_sql_query("""
    SELECT 100.0 * SUM(liquidados) / SUM(poblacion) AS porcentaje
    FROM cuadro5
    WHERE provincia NOT IN ('Total', 'Sin información')
""", con)

Da 19,51 %, igual que el Total del cuadro (19,5 %).

### 5.4 Tema nuevo: el `JOIN` que pierde filas sin avisar

**Predecir.** Unimos `cuadro5` con `provincias` para agregarle el código oficial a cada fila. ¿Cuántas filas devuelve el `INNER JOIN` por nombre?

In [ ]:
unidas = pd.read_sql_query("""
    SELECT c.provincia, p.id, c.liquidados
    FROM cuadro5 AS c
    INNER JOIN provincias AS p ON c.provincia = p.nombre
""", con)
print(len(unidas), "filas")

22, y no 24. El `INNER JOIN` **descarta sin avisar** las filas que no encuentran pareja. Para ver cuáles son, se usa un `LEFT JOIN`, que conserva todas las filas de la izquierda, y se buscan las que quedaron sin código. Es el equivalente de mirar `_merge` en Stata.

In [ ]:
pd.read_sql_query("""
    SELECT c.provincia, p.id
    FROM cuadro5 AS c
    LEFT JOIN provincias AS p ON c.provincia = p.nombre
    WHERE p.id IS NULL
""", con)

Quedan sin pareja `CABA` (en la tabla oficial, «Ciudad Autónoma de Buenos Aires») y `Tierra del Fuego` («Tierra del Fuego, Antártida e Islas del Atlántico Sur»), además de «Sin información» y «Total», que no son provincias y está bien que no tengan código. Piensen qué conclusión habrían sacado sobre la cobertura del IFE si la Ciudad desaparecía del análisis sin que nadie lo notara. Es el mismo problema de los nombres de la clase 4, y la razón por la que en la clase 6 vamos a unir por código.

**Ejercicio.** La solución que escala no es renombrar a mano dentro del cuadro, sino tener una tabla `nombres` con **todas** las formas en que aparece cada provincia, y su código. Créenla con los nombres oficiales más los dos alternativos, insertando estos dos con `?` (como en la slide 27), y repitan el `JOIN` contra `nombres`.

In [ ]:
con.execute("DROP TABLE IF EXISTS nombres")
con.execute("CREATE TABLE nombres AS SELECT nombre, id FROM provincias")
# inserten los dos nombres alternativos con executemany y "?"
con.commit()

con_codigo = pd.read_sql_query("""
    SELECT ...
""", con)

<details><summary><b>Ver una solución</b> (ábranla después de intentarlo)</summary>

```python
con.execute("DROP TABLE IF EXISTS nombres")
con.execute("CREATE TABLE nombres AS SELECT nombre, id FROM provincias")
con.executemany("INSERT INTO nombres (nombre, id) VALUES (?, ?)",
                [("CABA", "02"), ("Tierra del Fuego", "94")])
con.commit()

con_codigo = pd.read_sql_query("""
    SELECT n.id, c.*
    FROM cuadro5 AS c
    INNER JOIN nombres AS n ON c.provincia = n.nombre
    ORDER BY n.id
""", con)
print(len(con_codigo), "filas")
con_codigo.head(3)
```

**Por qué `?` y no un *f-string*.** El módulo se encarga de las comillas (un nombre como `O'Higgins` rompería la consulta) y evita la *inyección de SQL*, que un texto se ejecute como código. Si ejecutan dos veces el `INSERT` sin el `DROP TABLE IF EXISTS`, quedan nombres repetidos y el `JOIN` duplica filas: la verificación de abajo lo detecta.
</details>

Si no les salió, copien la solución en la celda de arriba y ejecútenla antes de seguir: las celdas siguientes la usan.

In [ ]:
# Verificación
assert len(con_codigo) == 24, f"Deberían quedar 24 provincias y hay {len(con_codigo)}"
assert con_codigo["id"].is_unique, "Algún código aparece dos veces: ¿insertaron un nombre repetido?"
assert con_codigo["id"].str.len().eq(2).all(), "Los códigos deberían tener dos caracteres ('02', no 2)"
print("✔ 24 provincias, cada una con su código del INDEC")

### 5.5 Lo mismo en pandas

Para comparar: `merge` con `validate` (frena si alguna clave está repetida) e `indicator` (agrega la columna `_merge`). Con `how="left"` se ven las filas que no encontraron pareja.

In [ ]:
nombres = pd.read_sql_query("SELECT * FROM nombres", con)
control = cuadro5.merge(nombres, left_on="provincia", right_on="nombre", how="left",
                        validate="one_to_one", indicator=True)
print(control["_merge"].value_counts())
control.loc[control["_merge"] == "left_only", "provincia"].tolist()

| Tarea | SQL | pandas | Stata |
| --- | --- | --- | --- |
| Elegir columnas | `SELECT a, b` | `df[["a", "b"]]` | `keep a b` |
| Filtrar filas | `WHERE x > 10` | `df[df["x"] > 10]` | `keep if x > 10` |
| Ordenar | `ORDER BY x DESC` | `df.sort_values("x", ascending=False)` | `gsort -x` |
| Agregar por grupo | `GROUP BY g` + `SUM(x)` | `df.groupby("g")["x"].sum()` | `collapse (sum) x, by(g)` |
| Unir | `INNER JOIN t ON ...` | `df.merge(t, on=..., how="inner")` | `merge 1:1 ... , keep(match)` |
| Ver los que no unieron | `LEFT JOIN ... WHERE t.id IS NULL` | `indicator=True` y `_merge == "left_only"` | `_merge == 1` |

¿Cuándo SQL y cuándo pandas? Si los datos ya están en una base (muchos registros administrativos, la base de una empresa) o son demasiado grandes para la memoria, la consulta se hace en SQL y solo el resultado pasa a pandas. Para bases grandes en su computadora existe también DuckDB, que hace consultas SQL directamente sobre archivos CSV o Parquet; no lo veremos a detalle en el curso, pero sepan que existen.

In [ ]:
con.close()

## 6. El trabajo final (TP6): empiecen a elegir sus datos

El TP6 es el **trabajo final** del curso y será individual. Este tiene una diferencia importante con los anteriores: **los datos los eligen ustedes.** La condición es que se puedan obtener con alguno de los métodos de esta clase:

| Método | Ejemplos de fuentes |
| --- | --- |
| Descarga directa de un archivo (CSV, Excel) | INDEC, portales de datos abiertos de ministerios, provincias o municipios |
| API | [APIs de datos.gob.ar](https://datos.gob.ar/apis) (series de tiempo, Georef), [APIs del BCRA](https://www.bcra.gob.ar/apis-banco-central/), [API de indicadores del Banco Mundial](https://datahelpdesk.worldbank.org/knowledgebase/articles/889392-about-the-indicators-api) |
| *Web scraping* | Precios publicados en tiendas en línea, listados de establecimientos, tablas de organismos sin descarga |
| PDF | Informes y boletines estadísticos de ministerios, ANSES, bancos centrales o cámaras empresarias |
| Base SQL | Una base que ya usen en su trabajo o su tesis, o una que armen con lo que descarguen |

Les recomiendo que **piensen en su tesis**: es la mejor oportunidad para dejar resuelta la obtención de datos que después van a necesitar.

**Antes de decidir, respondan estas preguntas sobre la fuente:**

1. **¿Qué pregunta quiero responder?** Una pregunta concreta, no un tema.
2. **¿Dónde están los datos?** La dirección exacta de la página, la API o el informe.
3. **¿Con qué método los obtengo?** Y si hay más de uno, ¿por qué elijo ese? Si existe una API o una descarga oficial, es mejor que *scrapear*.
4. **¿Puedo repetirlo?** Que el código pueda volver a bajar los datos, o que quede una copia cruda guardada en el repositorio.
5. **¿Puedo usarlos?** Términos de uso, licencia, `robots.txt`, claves de acceso, datos personales.
6. **¿Cómo voy a saber que están bien?** Un total publicado, una cifra que aparece en el texto, otra fuente para comparar.
7. **¿Qué tamaño tienen y qué período cubren?** Que alcance para la pregunta y que se pueda trabajar en su computadora.

En la clase 6 vamos a ver mapas: si sus datos tienen una dimensión geográfica (provincias, departamentos, coordenadas), guarden los **códigos oficiales como texto**, porque los van a necesitar para unirlos con una capa.

Las consignas completas del TP6 y su fecha de entrega llegan más adelante. **Por ahora, vayan definiendo la fuente**: completen esta ficha en una celda de texto y tráiganla a la próxima clase, o compártanla por Slack si quieren una devolución antes.

El TP5 consiste en la entrega de esta ficha de manera **individual**. La forma de entrega es mediante un repositorio individual de **GitHub** y el **la fecha de entrega es el miercoles 27/10 a las 6pm**

**Ficha de datos para el TP6** *(completar)*

- **Pregunta:**
- **Fuente (URL):**
- **Método de obtención:** descarga / API / *scraping* / PDF / SQL
- **Unidad de observación y período:**
- **Condiciones de uso o licencia:**
- **Cómo voy a verificar los datos:**
- **Para qué me sirve en la tesis (si aplica):**